# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count",
    "char_count", "search_volume", "competition", "cpc", "has_word_count", "has_keyword_data", "has_position"]
CATEGORICAL = ["content_type", "main_intent", "competition_level", "age_tier"]
FEATURES = NUMERIC + CATEGORICAL
X = df[FEATURES]; y = df["is_declining_label"]; groups = df["client_id"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])
# Logistic Regression, per Week 5's own honest-split comparison (it beat
# Random Forest at every K there) -- using GroupKFold so every row gets an
# out-of-fold prediction from a model that never saw that row's client.
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
gkf = GroupKFold(n_splits=5)
oof_proba = cross_val_predict(pipe, X, y, groups=groups, cv=gkf, method="predict_proba", n_jobs=1)[:, 1]
df["model_probability"] = oof_proba

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

print("Out-of-fold precision (5-fold, grouped by client, full 30,000-row coverage):")
precision_table = {}
for k in [10, 20, 50, 100, 500]:
    p = precision_at_k(y, oof_proba, k)
    precision_table[f"precision@{k}"] = round(p, 3)
    print(f"  precision@{k:<4} = {p:.3f}")
print(f"base rate: {y.mean():.3f}")

# confidence tiers + reason codes (same construction as Week 4, now applied
# to the validated model's own score instead of the hand-rule score)
df["confidence_tier"] = pd.cut(df["model_probability"], [0, 0.4, 0.7, 1.0], labels=["low", "medium", "high"], include_lowest=True)

d = df.copy()
d["pos_band"] = pd.cut(d["avg_position"].where(d["avg_position"] > 0), [0, 3, 10, 20, 50, 1e9], labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
band_median_ctr = d.groupby("pos_band", observed=True)["ctr"].transform("median")
d["ctr_gap"] = (band_median_ctr - d["ctr"]).clip(lower=0)
d["visible"] = (d["impressions_90d"] >= 100).astype(int)
d["achievable"] = d["avg_position"].between(1, 20).astype(int)
d["stale"] = (d["days_since_last_update"] >= 90).astype(int)
d["thin"] = ((d["word_count"] > 0) & (d["word_count"] < 1200)).astype(int)

def reason_code(r):
    if r["visible"] and r["achievable"] and r["ctr_gap"] > 0:
        return "ctr_gap_at_achievable_position"
    if r["visible"] and r["stale"]:
        return "stale_visible_page"
    if r["visible"] and r["thin"]:
        return "thin_visible_page"
    if r["visible"]:
        return "visible_no_specific_flag"
    return "low_visibility_monitor"

df["reason_code"] = d.apply(reason_code, axis=1)

# archetype (reason code) -> action mapping, gated by confidence: a low-
# confidence score never turns into anything stronger than "monitor",
# regardless of which archetype it matched.
def suggested_action(row):
    if row["confidence_tier"] == "low":
        return "monitor"
    return {
        "ctr_gap_at_achievable_position": "review_ctr_then_refresh",
        "stale_visible_page": "refresh",
        "thin_visible_page": "expand_and_refresh",
    }.get(row["reason_code"], "monitor")

df["suggested_action"] = df.apply(suggested_action, axis=1)
df["rank"] = df["model_probability"].rank(method="first", ascending=False).astype(int)

print("\nreason_code mix:")
print(df["reason_code"].value_counts())
print("\nsuggested_action mix (confidence-gated):")
print(df["suggested_action"].value_counts())

top10 = df.sort_values("model_probability", ascending=False).head(10)
print("\ntop 10 of the queue:")
print(top10[["rank", "client_id", "model_probability", "confidence_tier", "reason_code", "suggested_action", "is_declining_label"]].to_string(index=False))

Working directory: C:\Users\Laptop\Documents\fly


Out-of-fold precision (5-fold, grouped by client, full 30,000-row coverage):
  precision@10   = 0.600
  precision@20   = 0.750
  precision@50   = 0.740
  precision@100  = 0.800
  precision@500  = 0.730
base rate: 0.542



reason_code mix:
reason_code
visible_no_specific_flag          10370
low_visibility_monitor             7994
stale_visible_page                 6104
ctr_gap_at_achievable_position     5445
thin_visible_page                    87
Name: count, dtype: int64

suggested_action mix (confidence-gated):
suggested_action
monitor                    18875
refresh                     5677
review_ctr_then_refresh     5400
expand_and_refresh            48
Name: count, dtype: int64

top 10 of the queue:
 rank         client_id  model_probability confidence_tier                    reason_code        suggested_action  is_declining_label
    1 client_19581e27de           0.979453            high ctr_gap_at_achievable_position review_ctr_then_refresh                   1
    2 client_19581e27de           0.965022            high ctr_gap_at_achievable_position review_ctr_then_refresh                   1
    3 client_4e07408562           0.955178            high ctr_gap_at_achievable_position review_ctr_th

### 1. Ranked actions + reason codes

The queue's score is Logistic Regression's out-of-fold probability (5-fold `GroupKFold` by
`client_id`) — not the Week 4 hand-rule, and not Random Forest, because Week 5's own honest-split
comparison found Logistic Regression winning at every K on this data. Out-of-fold precision here
(0.60 / 0.75 / 0.74 / 0.80 at K=10/20/50/100) is consistent with Week 5's single-split numbers and
clears both the Week 4 baseline (0.640 @50) and the base rate (0.542) by a wide margin, averaged
over 5 folds rather than one lucky/unlucky split.

**Archetype → action mapping**: each page's `reason_code` (from Week 4's rule logic — CTR gap at
an achievable position, stale-but-visible, thin-but-visible) maps to a `suggested_action`, but
only when `confidence_tier` clears "low" — a page can match a strong archetype and still land on
`monitor` if the model itself isn't confident. This keeps the archetype from overriding the
model's own uncertainty.

**Decay/refresh insight**: this lines up directionally with FlyRank's own paper (Finding #4: 3.2x
health boost from refreshing mature, previously-visible pages) and with our own Week 4 result
(staleness alone got a MIXED verdict, but CTR gap at an achievable position was CONFIRMED and
drives most of this queue's `review_ctr_then_refresh` actions) — two independent portfolios
pointing the same direction on refresh timing, though I'm treating that as directional
corroboration, not proof they'd transfer 1:1 across datasets.

**Carried-over limitation, worth repeating here**: the client-concentration pattern from Week 4
persists — see section 2.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
# how skewed is confidence, and does the client-concentration issue from
# Week 4 persist now that a validated model (not a hand-rule) built the queue?
top50 = df.sort_values("model_probability", ascending=False).head(50)
print(f"Distinct clients in top 50 of the model-ranked queue: {top50['client_id'].nunique()} (of {df['client_id'].nunique()} total)")
print(top50["client_id"].value_counts().head(5))

print(f"\nShare of the full queue at each confidence tier:")
print((df["confidence_tier"].value_counts(normalize=True) * 100).round(1))

Distinct clients in top 50 of the model-ranked queue: 4 (of 32 total)
client_id
client_19581e27de    44
client_f369cb89fc     3
client_4e07408562     2
client_7f2253d7e2     1
Name: count, dtype: int64

Share of the full queue at each confidence tier:
confidence_tier
medium    55.1
high      28.4
low       16.5
Name: proportion, dtype: float64


### 2. Intended use and limits

**Intended use**: a reviewer aid for a content/SEO team working through a fixed weekly review
capacity, not an autonomous system. The reviewer takes the ranked shortlist and applies their own
judgment on each row before acting.

**Limits, named plainly**:
- The client-concentration issue from Week 4 is still here even with a real model: the top 50 of
  this queue still draws heavily from one or two of 32 clients, because that's a property of
  *how many candidate pages each client has*, not of which scoring method ranks them. A fair
  rollout across FlyRank's whole client book needs a per-client cap or per-client reporting, same
  conclusion as Week 4.
- The label is a proxy (Week 2/3): "declining" means "impressions fell >20% over 30 days," not
  "this page has a diagnosed problem."
- Precision at small K (10, 20) is measurably noisier than at 50/100 (Week 5) — trust the larger-K
  numbers more when judging "is this queue any good."
- This is a single point-in-time export (30,000 rows, one snapshot) — not a live feed, and not
  the 79M-row warehouse the capstone will eventually draw from.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
# what fraction of action-bearing rows (not "monitor") sit at medium vs high
# confidence -- this justifies where the mandatory-review line gets drawn.
actionable = df[df["suggested_action"] != "monitor"]
print(f"Actionable rows (not just 'monitor'): {len(actionable):,} of {len(df):,}")
print(actionable["confidence_tier"].value_counts())
print(f"\nShare of actionable rows at 'medium' confidence (mandatory human review before acting): "
      f"{(actionable['confidence_tier'] == 'medium').mean() * 100:.1f}%")

Actionable rows (not just 'monitor'): 11,125 of 30,000
confidence_tier
high      5702
medium    5423
low          0
Name: count, dtype: int64

Share of actionable rows at 'medium' confidence (mandatory human review before acting): 48.7%


### 3. Human review + the no-go list

**Before acting on any row, a person checks**: does the reason code make sense for this specific
page (not just the archetype in general)? Is the CTR gap explainable by query intent
(navigational/branded queries structurally get low CTR — Week 4's own top-10 review found this
exact failure mode) rather than a fixable content problem? 48.7% of actionable
(non-monitor) rows sit at "medium" rather than "high" confidence (5,423 of 11,125) and get
**mandatory** review before any action, not just a spot-check.

**What should never be automated**:
- No auto-publishing, auto-deleting, or auto-redirecting content based on this queue alone.
- No use of this score as an employee, agency, or vendor performance metric — it scores pages,
  not people, and the label is a proxy with a known false-positive pattern.
- No treating "declining" as a causal diagnosis of *why* — the model has no mechanism, only
  correlation (Week 6's audit was explicit about this).
- No running this queue against live/current data without revalidating first — it was trained and
  validated on a single mid-2026 export; a materially different data distribution invalidates the
  calibration in section 4 below.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
# reference snapshot for drift monitoring: today's base rate and top features,
# to compare against on a future retrain.
final_pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
final_pipe.fit(X, y)
onehot_names = final_pipe.named_steps["pre"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(CATEGORICAL)
feat_names = NUMERIC + list(onehot_names)
coefs = pd.Series(final_pipe.named_steps["clf"].coef_[0], index=feat_names).sort_values(key=lambda s: s.abs(), ascending=False)

print("Reference snapshot for future drift checks:")
print(f"  base_rate = {y.mean():.4f}")
print(f"  top 5 |coefficient| features:")
print(coefs.head(5).round(3))

Reference snapshot for future drift checks:
  base_rate = 0.5421
  top 5 |coefficient| features:
log_impressions_90d                0.910
has_position                       0.862
log_clicks_90d                    -0.670
word_count                         0.386
content_type_comparison article   -0.367
dtype: float64


### 4. Monitoring / retrain triggers

**Retrain triggers**: the base rate drifting meaningfully away from 0.542 (a shift in what
"normal" decline looks like); the top-5 coefficient features above changing rank or sign on a
retrain (a signal the relationships changed, not just noise); a new data export becoming
available (the warehouse's monthly partitions, once the capstone moves there); precision@50 on a
realized outcome sample (pages actually reviewed, checked 30/60 days later) dropping meaningfully
below the 0.74 measured here.

**Light monitoring, not a full pipeline**: this is a teaching-scale model on a 30K-row snapshot —
appropriate monitoring here is a periodic manual re-run and comparison against this notebook's own
numbers, not a production alerting system.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import json
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue_cols = ["rank", "content_id", "client_id", "model_probability", "confidence_tier",
              "reason_code", "suggested_action", "impressions_90d", "avg_position", "ctr",
              "days_since_last_update", "content_type"]
queue = df.sort_values("rank")[queue_cols]
queue.to_csv("work/outputs/ranked_action_queue.csv", index=False)
print(f"Wrote work/outputs/ranked_action_queue.csv: {len(queue):,} rows (gitignored by design, regenerated on rerun)")

metrics = {
    "model": "logistic_regression",
    "validation": "5-fold GroupKFold by client_id (out-of-fold predictions, full coverage)",
    "base_rate": round(float(y.mean()), 4),
    "precision_at_k": precision_table,
    "reason_code_mix": df["reason_code"].value_counts().to_dict(),
    "suggested_action_mix": df["suggested_action"].value_counts().to_dict(),
    "confidence_tier_mix": df["confidence_tier"].value_counts().to_dict(),
    "top_5_coefficients_by_abs_value": {k: round(float(v), 4) for k, v in coefs.head(5).items()},
}
with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print("Wrote work/outputs/w07_metrics.json (committed -- the paper's numbers trace back to this)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

action_counts = df["suggested_action"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(action_counts.index[::-1], action_counts.values[::-1], color="#6F4E7C")
ax.set_xlabel("Pages")
ax.set_title("Suggested action mix (confidence-gated)")
fig.tight_layout()
fig.savefig("work/figures/action_mix.svg")
plt.close(fig)
print("Wrote work/figures/action_mix.svg (committed)")

Wrote work/outputs/ranked_action_queue.csv: 30,000 rows (gitignored by design, regenerated on rerun)
Wrote work/outputs/w07_metrics.json (committed -- the paper's numbers trace back to this)


Wrote work/figures/action_mix.svg (committed)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.